In [ ]:
!pip install pyspark

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving employees_updated.csv to employees_updated.csv


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("SparkSQL") \
    .getOrCreate()

In [ ]:
df = spark.read.csv("employees_updated.csv", header=True, inferSchema=True)
df.createOrReplaceTempView("employees")

df.show()

+----------+---------+----------+--------------------+-----------+----------+----------+------------+----------------+-----+--------------------+--------+----------+--------------------+--------+----------+
|EmployeeID|FirstName|  LastName|               Email|PhoneNumber|  HireDate|Commission|DepartmentID|  DepartmentName|JobID|            JobTitle|  salary|LocationID|     PhysicalAddress|Postcode|      City|
+----------+---------+----------+--------------------+-----------+----------+----------+------------+----------------+-----+--------------------+--------+----------+--------------------+--------+----------+
|     79237|     Joan|     Bryan| jamie26@example.org|+44(0)20 74|2017-09-15|       0.0|          10|              HR|    5| IT Security Officer|37630.71|        78|Studio 91 Claire ...| S39 9TX|Colchester|
|     95133|    Colin|  Thompson|  xallen@example.com|+44(0)11549|2021-11-18|       0.0|          15|         Finance|    5| IT Security Officer|37630.71|        72|    873

In [18]:
#1: Count employees hired in 2021
spark.sql("""
SELECT COUNT(*) AS total_2021
FROM employees
WHERE YEAR(HireDate) = 2021
""").show()

+----------+
|total_2021|
+----------+
|      6828|
+----------+



In [19]:
#2: Employees working in address containing 'close'
spark.sql("""
SELECT FirstName, LastName, PhysicalAddress
FROM employees
WHERE LOWER(PhysicalAddress) LIKE '%close%'
""").show()

+---------+---------+---------------+
|FirstName| LastName|PhysicalAddress|
+---------+---------+---------------+
|   Teresa|   Hughes|168 River Close|
|    Molly|    Kirby|168 River Close|
|  Suzanne|   Thomas|168 River Close|
|    Conor|  Jackson|168 River Close|
|     Tina|    Doyle|168 River Close|
|    Holly| Robinson|168 River Close|
|  Eleanor|   Jordan|168 River Close|
|     Tina|    Mason|168 River Close|
|   Joshua|    Smith|168 River Close|
|   Cheryl|     Shah|168 River Close|
|  Natasha|    Ellis|168 River Close|
|    Conor|   Wright|168 River Close|
|  Stewart|   Turner|168 River Close|
|  Marilyn|  Simpson|168 River Close|
|   Ashley|     Read|168 River Close|
|   Elliot|Whittaker|168 River Close|
|   Stacey|   Coates|168 River Close|
|   Andrea|    Parry|168 River Close|
| Margaret|   Abbott|168 River Close|
| Nicholas|   Harper|168 River Close|
+---------+---------+---------------+
only showing top 20 rows


In [20]:
#3: Job title with highest total salary cost
spark.sql("""
SELECT JobTitle, ROUND(SUM(salary), 2) AS total_cost
FROM employees
GROUP BY JobTitle
ORDER BY total_cost DESC
LIMIT 1
""").show()

+--------+--------------+
|JobTitle|    total_cost|
+--------+--------------+
|     CEO|4.4954852425E8|
+--------+--------------+



In [21]:
#4: Total salary after adding commission
spark.sql("""
SELECT FirstName, LastName,
       salary + COALESCE(Commission, 0) AS total_salary
FROM employees
""").show()

+---------+----------+------------------+
|FirstName|  LastName|      total_salary|
+---------+----------+------------------+
|     Joan|     Bryan|          37630.71|
|    Colin|  Thompson|          37630.71|
|   Norman|  Lawrence|           64829.3|
|  Heather|     Lloyd|          53191.23|
|    Abdul|   Johnson|          39652.76|
|   Thomas|   Brookes|           39681.2|
|   Jeremy|    Wilson|          88581.47|
|   Denise|     Ellis|          50213.78|
| Clifford|   Cameron|          39652.76|
|    Billy|  Robinson|39653.590000000004|
|  Douglas|     Begum| 88581.59000000001|
|   Cheryl|    Barker|           39681.2|
|   Sheila|     Field|          54965.17|
| Rosemary|    Curtis|          88581.46|
|   Trevor|  Williams|          42860.49|
|   Connor|  Reynolds|           64829.3|
|  Charlie|      Reid|          37630.71|
|     Tina|  Marshall|          48609.99|
|  Stewart|Cartwright|          50213.78|
|    Clive|    Taylor|           39681.2|
+---------+----------+------------

In [22]:
#5: Most common hiring day of the month
spark.sql("""
SELECT DAY(HireDate) AS day, COUNT(*) AS total
FROM employees
GROUP BY day
ORDER BY total DESC
LIMIT 1
""").show()

+---+-----+
|day|total|
+---+-----+
|  5| 1733|
+---+-----+



In [23]:
#6: Department with highest number of employees
spark.sql("""
SELECT DepartmentName, COUNT(*) AS total
FROM employees
GROUP BY DepartmentName
ORDER BY total DESC
LIMIT 1
""").show()

+--------------+-----+
|DepartmentName|total|
+--------------+-----+
|       Finance| 6369|
+--------------+-----+



In [24]:
#7: Average salary in each department
spark.sql("""
SELECT DepartmentName, ROUND(AVG(salary), 2) AS avg_salary
FROM employees
GROUP BY DepartmentName
""").show()

+----------------+----------+
|  DepartmentName|avg_salary|
+----------------+----------+
|           Sales|  52078.27|
|Customer Support|   52085.4|
|              HR|  52299.61|
|         Finance|   52171.5|
|        Research|  51963.85|
|       Marketing|  51804.88|
|          Design|  51942.49|
|  Administration|  51944.01|
+----------------+----------+



In [25]:
#8: Employees working for more than 5 years
spark.sql("""
SELECT COUNT(*) AS total
FROM employees
WHERE months_between(current_date(), HireDate) > 60
""").show()

+-----+
|total|
+-----+
|29853|
+-----+



In [26]:
#9: Total commission paid per department
spark.sql("""
SELECT DepartmentName, SUM(Commission) AS total_commission
FROM employees
GROUP BY DepartmentName
""").show()

+----------------+-----------------+
|  DepartmentName| total_commission|
+----------------+-----------------+
|           Sales|610.2599999999999|
|Customer Support|612.5399999999998|
|              HR|631.8799999999999|
|         Finance|629.7200000000004|
|        Research|610.9700000000008|
|       Marketing|592.1299999999999|
|          Design|614.6599999999996|
|  Administration|609.0200000000001|
+----------------+-----------------+



In [27]:
#10: Employee with highest total salary after commission
spark.sql("""
SELECT FirstName, LastName,
       salary + COALESCE(Commission, 0) AS total_salary
FROM employees
ORDER BY total_salary DESC
LIMIT 1
""").show()

+---------+--------+------------+
|FirstName|LastName|total_salary|
+---------+--------+------------+
|  Dorothy|  Haynes|    88581.99|
+---------+--------+------------+

